# Sistemas de Preguntas y Respuestas con Transformers (**Student Version**)

**Task:** Question Answering (QA) sobre contexto dado + mini QA "open-domain" usando un modelo preentrenado de `transformers`.

**Modelo sugerido:** `deepset/roberta-base-squad2` (inglés, extractive QA).  
Puedes cambiar de modelo si lo prefieres, siempre que sea compatible con `pipeline("question-answering")`.

---

## Objetivos del notebook

En este notebook vas a:

1. Cargar un modelo de **QA extractivo** preentrenado usando `transformers`.
2. Probar el modelo en ejemplos sencillos de "pregunta + contexto" (tipo SQuAD).
3. Implementar una mini-evaluación con **Exact Match (EM)** y **F1** a nivel de tokens en un pequeño dataset manual.
4. Implementar una mini QA "open-domain" sobre un **corpus pequeño** de párrafos, usando un método sencillo de recuperación de contexto.
5. Analizar errores típicos y limitaciones del enfoque.

---

## Qué se asume que ya sabes

- Qué es QA extractivo frente a QA generativo (Tema 4).
- Nociones de modelos BERT/Roberta finetuneados en SQuAD.
- Python y nociones básicas de `transformers`.


## 1) (Opcional) Instalación / actualización de librerías

**Qué hacer:**  
Si estás en Google Colab o en un entorno donde no tienes las librerías instaladas (o quieres actualizar), puedes usar la celda de abajo.

- Si ya tienes el entorno preparado (p. ej. proporcionado por el profesor), puedes **saltar esta celda**.


In [ ]:
# TODO (opcional): instala o actualiza las librerías necesarias.
# Ejemplo orientativo (NO lo dejes comentado si lo necesitas):
# !pip install -q "transformers>=4.40.0"


## 2) Imports y configuración básica

**Qué hacer:**

1. Importar:
   - `pipeline` de `transformers` (lo usaremos para QA).
   - `re` y, si quieres, `numpy` para pequeñas utilidades.
2. Comprobar que todo se importa sin errores.


In [ ]:
# TODO: importa las librerías necesarias.
# Pistas:
# from transformers import pipeline
# import re
# import numpy as np  # opcional, para cálculos de medias, etc.


## 3) Creación del pipeline de QA extractivo

Vamos a crear un `pipeline` de tipo `"question-answering"`.

**Qué hacer:**

1. Crea un objeto `qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")`.
2. Haz una prueba rápida con un ejemplo pequeño:
   - Define un `context` (2–3 frases).
   - Define una `question` sobre ese contexto.
   - Llama a `qa_pipeline({"question": ..., "context": ...})`.
3. Imprime el resultado y mira los campos que devuelve (`answer`, `score`, etc.).


In [ ]:
# TODO: crea el pipeline de QA y haz una prueba rápida.
# Pistas:
# qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")
#
# context = """Your context text here..."""
# question = "Your question here?"
# result = qa_pipeline({"question": question, "context": context})
# print(result)


## 4) Definición de ejemplos de contexto + preguntas

Primero vamos a trabajar en modo "QA sobre contexto dado", es decir, **closed-book** con un párrafo concreto.

**Qué hacer:**

1. Define al menos **dos contextos** (párrafos) en inglés, por ejemplo sobre:
   - NLP / IA.
   - Otro tema que te interese (historia, ciencia, etc.).
2. Para cada contexto, define varias preguntas que puedan responderse con un fragmento exacto del texto.
3. Guarda todo en una estructura tipo:

```python
qa_examples = [
    {
        "context": "...",
        "question": "...",
        "answer": "..."  # respuesta correcta (span dentro del contexto)
    },
    ...
]
```
4. Imprime algunos ejemplos para comprobar que se han guardado bien.


In [ ]:
# TODO: define una lista `qa_examples` con varios diccionarios
# con las claves: "context", "question", "answer".
#
# Ejemplo de estructura (rellena con contenido real):
# qa_examples = [
#     {
#         "context": """Natural language processing (NLP) is a subfield of artificial
#         intelligence that focuses on the interaction between computers and humans
#         through natural language.""".strip(),
#         "question": "What does NLP focus on?",
#         "answer": "the interaction between computers and humans through natural language"
#     },
#     ...
# ]
#
# TODO: imprime al menos un par de ejemplos para revisar.


## 5) Función auxiliar `answer_question`

Para no repetir código, vamos a definir una función que envuelva el pipeline.

**Qué hacer:**

1. Implementa una función `answer_question(question, context)` que:
   - Reciba `question` (string) y `context` (string).
   - Llame internamente a `qa_pipeline`.
   - Devuelva el diccionario de salida del pipeline (o sólo `answer`, si lo prefieres, pero mejor el diccionario completo).
2. Prueba la función con algún ejemplo de `qa_examples`.


In [ ]:
# TODO: implementa `answer_question(question, context)` envolviendo el pipeline.
# Pistas:
# def answer_question(question, context):
#     return qa_pipeline({"question": question, "context": context})
#
# TODO: prueba la función con un ejemplo de `qa_examples[0]`.


## 6) Exploración de `answer`, `score` y spans

El pipeline de QA devuelve típicamente algo como:

```python
{
  'score': 0.87,
  'start': 42,
  'end': 84,
  'answer': '...'
}
```

**Qué hacer:**

1. Recorre tus `qa_examples` y, para cada uno, llama a `answer_question`.
2. Imprime:
   - La pregunta.
   - La respuesta predicha (`pred["answer"]`).
   - El `score`.
   - La respuesta correcta (`example["answer"]`).
3. Observa casos donde el modelo:
   - Acierte claramente.
   - Falle por poco (elige un span casi correcto).
   - Falle de forma clara.


In [ ]:
# TODO: recorre `qa_examples` y muestra la predicción del modelo
# (answer + score) junto con la respuesta correcta.


## 7) Normalización de texto y métricas EM / F1

Vamos a implementar una evaluación sencilla tipo SQuAD:

- **Exact Match (EM):** 1 si la respuesta predicha coincide exactamente (después de normalizar) con la respuesta correcta, 0 en caso contrario.
- **F1 token-level:** medida de solapamiento entre los tokens de la predicción y los de la respuesta correcta.

**Qué hacer:**

1. Implementa una función `normalize_text(s)` que:
   - Pase a minúsculas.
   - Elimine signos de puntuación simples.
   - Elimine espacios extra.
2. Implementa:
   - `compute_exact_match(pred, gold)` → 1 o 0.
   - `compute_f1(pred, gold)` → valor en [0, 1] basado en tokens.
3. Prueba estas funciones con ejemplos sencillos (por ejemplo, variando mayúsculas, puntuación, etc.).


In [ ]:
# TODO: implementa normalize_text, compute_exact_match y compute_f1.
# Pistas:
# - Usa re.sub(...) para quitar puntuación.
# - Para F1, tokeniza en palabras (split) y calcula:
#   precision = (# tokens en común) / (# tokens pred)
#   recall    = (# tokens en común) / (# tokens gold)
#   f1 = 2 * precision * recall / (precision + recall)  (con caso especial si ambos son 0).


## 8) Evaluación en tu mini-dataset de QA

Ahora aplicamos las métricas EM y F1 a los ejemplos que has definido.

**Qué hacer:**

1. Para cada ejemplo en `qa_examples`:
   - Obtén la predicción del modelo (span de respuesta).
   - Calcula EM y F1 comparando predicción vs `example["answer"]`.
2. Guarda los resultados en una lista o array.
3. Calcula EM medio y F1 medio sobre todos los ejemplos.
4. Imprime un pequeño resumen por ejemplo y un resumen global.

**Mini-análisis (en markdown):**  
¿En qué tipo de preguntas el modelo funciona mejor? ¿En cuáles falla más (fechas, entidades, definiciones, etc.)?


In [ ]:
# TODO: recorre `qa_examples`, llama a answer_question y evalúa con EM y F1.
# Pistas:
# - Para cada ejemplo, guarda un diccionario con:
#   {"pred": pred_answer, "gold": gold_answer, "em": em, "f1": f1}
# - Al final, calcula la media de EM y la media de F1.


## 9) Mini QA "open-domain" con recuperación sencilla

En QA open-domain, el sistema primero debe **encontrar** un contexto relevante y luego responder la pregunta.

Aquí vamos a hacerlo en versión "juguete" con un corpus pequeño de párrafos.

**Qué hacer:**

1. Define una lista `corpus` con varios párrafos sobre temas distintos (3–6 párrafos).
2. Define varias preguntas que se puedan contestar a partir de esos párrafos, pero sin indicar explícitamente cuál es el contexto para cada pregunta.
3. Guarda las preguntas en una lista `open_questions` y, si quieres, también las respuestas correctas en otra estructura para evaluar.


In [ ]:
# TODO: define un pequeño corpus de párrafos (lista de strings) y una lista de preguntas.
# Opcional: guarda también las respuestas correctas asociadas a cada pregunta.


## 10) Recuperación de contexto: escoger el mejor párrafo

Vamos a implementar un método muy simple de recuperación basado en **solapamiento de palabras** entre pregunta y párrafo.

Idea básica:
- Tokenizar pregunta y cada párrafo.
- Calcular, para cada párrafo, cuántas palabras comparten con la pregunta.
- Escoger el párrafo con mayor solapamiento.

**Qué hacer:**

1. Implementa una función `score_overlap(question, doc)` que devuelva un número (más alto = más similar).
2. Implementa `retrieve_best_context(question, corpus)` que:
   - Calcule `score_overlap(question, doc)` para cada doc en el corpus.
   - Devuelva el doc con mayor score (y si quieres, también el índice).
3. Prueba estas funciones con varias preguntas y comprueba si te devuelve el párrafo correcto.


In [ ]:
# TODO: implementa score_overlap(question, doc) y retrieve_best_context(question, corpus).
# Pistas:
# - Puedes reutilizar parte de la lógica de normalize_text / tokenización.
# - Una métrica sencilla: número de palabras en común entre conjuntos de tokens.


## 11) QA "open-domain" sobre tu mini-corpus

Ahora combinamos recuperación + QA:

1. Para cada pregunta en `open_questions`:
   - Usamos `retrieve_best_context` para obtener el contexto más probable.
   - Llamamos a `answer_question(question, best_context)`.
2. Imprimimos:
   - La pregunta.
   - El contexto seleccionado (o sus primeras frases).
   - La respuesta del modelo.

**Opcional:** Si has definido respuestas correctas para `open_questions`, puedes comparar la respuesta del modelo con ellas (usando EM/F1).


In [ ]:
# TODO: implementa el bucle de QA open-domain sobre tu mini-corpus.
# Pistas:
# - Para cada pregunta:
#     best_ctx = retrieve_best_context(question, corpus)
#     pred = answer_question(question, best_ctx)
# - Imprime la pregunta, parte del contexto seleccionado y la respuesta.


## 12) Conclusiones

En esta sección debes escribir un breve resumen (5–10 líneas) con tus conclusiones sobre:

- En qué se diferencia QA sobre contexto dado vs QA open-domain.
- Cómo afectan la calidad del contexto y la recuperación al rendimiento global del sistema.
- Qué tipo de errores has observado (respuestas vacías, spans demasiado largos/cortos, respuestas a la pregunta equivocada, etc.).
- Cómo se relacionan tus observaciones con los apartados 4.9–4.11 del Tema 4.

Puedes proponer también mejoras o extensiones: usar embeddings para la recuperación, probar otro modelo de QA, etc.
